# EDA — Crops NPK Dataset

**Project Terra** — Exploración inicial y verificación de calidad de datos.

Dataset: [Crops NPK Dataset (Kaggle)](https://www.kaggle.com/datasets/javakhan/crops-npk-data-set)

Columnas esperadas:
- `Nitrogen (N)`, `Phosphorus (P)`, `Potassium (K)` — mg/kg
- `Temperature` — °C
- `Humidity` — %
- `pH_Value` — escala 4.5–8.5
- `Rainfall` — mm
- `Crop` — cultivo (categórica)
- `Soil_Type` — tipo de suelo (categórica)
- `Variety` — variedad del cultivo (categórica)

> Coloca el CSV descargado de Kaggle en `data/raw/` y ajusta `DATA_PATH` en la siguiente celda si el nombre del archivo es distinto.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_PATH = "../data/raw/sensor_Crop_Dataset.csv"  # Ruta corregida al dataset real


## 1. Carga de datos

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()


In [ ]:
df.info()


## 2. Verificación de valores nulos y duplicados

In [ ]:
nulos = df.isnull().sum()
porcentaje_nulos = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({
    "nulos": nulos,
    "% nulos": porcentaje_nulos
}).sort_values("nulos", ascending=False)

resumen_nulos


In [ ]:
# Mapa de calor de valores nulos (útil si hay columnas con nulos dispersos)
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Mapa de valores nulos")
plt.show()


In [ ]:
duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {duplicados}")


## 3. Estadísticos descriptivos

In [ ]:
df.describe().T


In [ ]:
# Columnas categóricas: conteo de valores únicos
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    print(f"\n{col}: {df[col].nunique()} valores únicos")
    print(df[col].value_counts().head(10))


## 4. Distribución de variables numéricas

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()

fig, axes = plt.subplots(nrows=(len(num_cols) + 1) // 2, ncols=2, figsize=(14, 4 * ((len(num_cols) + 1) // 2)))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="seagreen")
    axes[i].set_title(f"Distribución de {col}")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


## 5. Detección de valores atípicos (boxplots)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=len(num_cols), figsize=(4 * len(num_cols), 5))

for i, col in enumerate(num_cols):
    sns.boxplot(y=df[col], ax=axes[i], color="lightcoral")
    axes[i].set_title(col)

plt.tight_layout()
plt.show()


## 6. Matriz de correlación

In [ ]:
plt.figure(figsize=(9, 7))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriz de correlación — variables numéricas")
plt.show()


## 7. Relación entre variables (pairplot)

In [ ]:
sample = df.sample(min(500, len(df)), random_state=42)  # muestra para que el pairplot no sea muy pesado
sns.pairplot(sample[num_cols], diag_kind="kde", plot_kws={"alpha": 0.5, "s": 20})
plt.suptitle("Pairplot de variables numéricas", y=1.02)
plt.show()


## 8. Distribución de variables categóricas

In [ ]:
if "Crop" in df.columns:
    plt.figure(figsize=(12, 6))
    order = df["Crop"].value_counts().index
    sns.countplot(data=df, y="Crop", order=order, palette="viridis")
    plt.title("Cantidad de registros por cultivo")
    plt.tight_layout()
    plt.show()


In [ ]:
if "Soil_Type" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df, x="Soil_Type", order=df["Soil_Type"].value_counts().index, palette="crest")
    plt.title("Cantidad de registros por tipo de suelo")
    plt.tight_layout()
    plt.show()


## 9. Conclusiones preliminares del EDA

- **Valores nulos**: No hay valores nulos en el dataset.
- **Filas duplicadas**: No hay filas duplicadas.
- **Atípicos (Outliers)**: Dependiendo de los boxplots, pero en base a la inspección estadística, se sugiere analizar a fondo variables como Rainfall o Temperature que pueden tener valores extremos.
- **Correlación fuerte (>0.6 o <-0.6)**: No hay pares con correlación > 0.6 o < -0.6.
- **Balance del dataset**: 
  - Top cultivos (proporción): {'Wheat': 0.1695, 'Potato': 0.1681, 'Maize': 0.1676}
  - Top tipos de suelo (proporción): {'Silt': 0.1711, 'Clay': 0.17005, 'Saline': 0.16825}
  - Conclusión: El dataset parece tener cierto desbalance que debe ser considerado.
